In [112]:
%pip install google-cloud-bigquery google-auth pandas pyarrow db_dtypes

/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
Note: you may need to restart the kernel to use updated packages.


In [113]:
import json
import time
from pathlib import Path
from typing import Any, Dict, Optional

import pandas as pd
import requests
from datasets import load_dataset
from tqdm.auto import tqdm

import csv
from google.cloud import bigquery
from google.oauth2 import service_account

In [114]:
DATASET_NAME = "xlangai/spider2-lite"
SPLIT = "train" # validation, train

ASK_URL = "http://localhost:8081/ask"
REQUEST_TIMEOUT_SECONDS = 120
SLEEP_BETWEEN_REQUESTS_SECONDS = 0.0

MAX_ROWS: Optional[int] = 100
OFFSET: Optional[int] = 0
BASE_OUTPUT_PATH: str = Path("spider2")

BASE_PAYLOAD: Dict[str, Any] = {
    "user_id": "user_123",
    "model": "google/gemma-4-E4B-it",
    "first_user_message": "I want to ask a question about my database schema.",
    "question": None,
    "is_thinking": None,
    "skip_user_interaction": True,
    "schema_context_base": None,
}

In [115]:
credentials = service_account.Credentials.from_service_account_file(
    "./config/bigquery_credential.json"
)

client = bigquery.Client(
    credentials=credentials,
    project=credentials.project_id,  # or "YOUR_GCP_PROJECT_ID"
)

In [116]:
dataset = load_dataset(DATASET_NAME, split=SPLIT)
df = dataset.to_pandas()

print(f"Loaded {len(df):,} rows from {DATASET_NAME}/{SPLIT}")
print("Columns:", list(df.columns))
df.head()

Loaded 260 rows from xlangai/spider2-lite/train
Columns: ['instance_id', 'db', 'question', 'external_knowledge', 'temporal']


,instance_id,db,question,external_knowledge,temporal
0,bq011,ga4,How many pseudo users were active in the last ...,ga4_obfuscated_sample_ecommerce.events.md,None
1,bq010,ga360,Find the top-selling product among customers w...,google_analytics_sample.ga_sessions.md,None
2,bq001,ga360,I wonder how many days between the first trans...,google_analytics_sample.ga_sessions.md,None
3,bq008,ga360,What's the most common next page for visitors ...,google_analytics_sample.ga_sessions.md,None
4,bq268,ga360,Identify the longest number of days between th...,None,None


In [117]:
def get_task_type(row):
    task_type = None
    if row["instance_id"].startswith("bq") or row["instance_id"].startswith("ga"):
        task_type = 'bq'
        row['type'] = 'Bigquery'
    elif row["instance_id"].startswith("local"):
        task_type = 'local'
        row['type'] = 'Local'
    elif row["instance_id"].startswith("sf"):
        task_type = 'sf'
        row['type'] = 'Snowflake'
    else:
        task_type = 'dbt'
    return task_type

In [118]:
def get_db_schema(dataset_name: str, table_name: str) -> str:
    sql_ddl_schema = f"""
    SELECT ddl
    FROM `bigquery-public-data.{dataset_name}.INFORMATION_SCHEMA.TABLES`
    WHERE table_name LIKE '{table_name}_%'
    LIMIT 1
    """

    rows = client.query(sql_ddl_schema, location="US").result()

    row = next(iter(rows), None)

    if row is None:
        return ""

    return row["ddl"]

In [119]:
def build_payload(question: str, schema: str, thinking: bool) -> Dict[str, Any]:
    """Create the /ask request body for one Spider row."""
    payload = dict(BASE_PAYLOAD)
    payload["is_thinking"] = thinking
    payload["question"] = question
    payload["schema_context_base"] = schema
    return payload

In [120]:
def post_question(question: str, schema: str, thinking: bool) -> Dict[str, Any]:
    """POST one row to /ask and return a normalized result dictionary."""
    payload = build_payload(question, schema, thinking)

    try:
        response = requests.post(
            ASK_URL,
            json=payload,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )

        result: Dict[str, Any] = {
            "request_payload": payload,
            "http_status_code": response.status_code,
            "request_error": None,
        }

        try:
            response_json = response.json()
        except ValueError:
            response_json = None
            result["raw_response_text"] = response.text

        result["response_json"] = response_json

        if isinstance(response_json, dict):
            result["ask_status"] = response_json.get("status")
            result["ask_answer"] = response_json.get("answer")
            result["ask_reached_end"] = response_json.get("reached_end")
        else:
            result["ask_status"] = None
            result["ask_answer"] = None
            result["ask_reached_end"] = None

        return result

    except requests.RequestException as exc:
        return {
            "request_payload": payload,
            "http_status_code": None,
            "request_error": repr(exc),
            "response_json": None,
            "ask_status": None,
            "ask_answer": None,
            "ask_reached_end": None,
        }

In [121]:
def get_table_and_schema_name(source: str):
    base = source.removesuffix(".md")

    # Split dataset and table name
    schema = base.split(".", maxsplit=1)
    return schema

In [122]:
def get_schema(task_type, row):
    schema = ""
    if task_type == "bq":
        dataset_name, table_name = get_table_and_schema_name(row["external_knowledge"])
        schema = get_db_schema(dataset_name, table_name)
    return schema

In [123]:
def execute_bq_query_to_csv(
    query: str,
    output_csv_path: str,
    location: str = "US",
) -> Path:
    output_path = Path(output_csv_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    query_job = client.query(query, location=location)
    rows = query_job.result()

    field_names = [field.name for field in rows.schema]

    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        # Header
        writer.writerow(field_names)

        # Rows
        for row in rows:
            writer.writerow([row.get(field) for field in field_names])

    return output_path

In [124]:
def save_sql_to_file(
    sql: str,
    output_path
) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)

    output_path.write_text(sql.strip() + "\n", encoding="utf-8")

    return output_path

In [125]:
def get_output_path(instance_id: str, suffix: str = ".csv") -> Path:
    suffix = suffix if suffix.startswith(".") else f".{suffix}"
    return BASE_OUTPUT_PATH / f"{instance_id}{suffix}"

In [126]:
def get_data_result(task_type, query, instance_id):
    output_path_csv = get_output_path(instance_id)
    output_path_sql = get_output_path(query, instance_id, ".sql")
    save_sql_to_file(query, output_path_sql)

    if task_type == "bq":
        execute_bq_query_to_csv(query, output_path_csv)

In [127]:
def ask_llm():
    work_df = df.copy()
    if MAX_ROWS is not None:
        work_df = work_df.iloc[OFFSET:OFFSET + MAX_ROWS].copy()

    results_thinking = []

    for _, row in tqdm(work_df.iterrows(), total=len(work_df), desc="POST /ask"):
        task_type = get_task_type(row)
        schema = get_schema(task_type, row)

        result = post_question(row["question"], schema, thinking=True)

        get_data_result(task_type, result, row["instance_id"])

        results_thinking.append(result)

        if SLEEP_BETWEEN_REQUESTS_SECONDS > 0:
            time.sleep(SLEEP_BETWEEN_REQUESTS_SECONDS)

    # Add both response sets to the original rows with clear column names
    work_df["ask_response_thinking"] = results_thinking

    results_df = work_df

    print(f"Collected {len(results_df):,} rows with thinking and non-thinking responses")
    return results_df

In [128]:
# Run this cell first to verify your local service is running and the payload shape is correct.
# Comment it out or skip it once verified.

sample_row = df.iloc[0]
task_type = get_task_type(sample_row)
schema = get_schema(task_type, sample_row)

sample_result = post_question(sample_row["question"], schema, True)
print(json.dumps(sample_result, indent=2, ensure_ascii=False))

instance_id                                                       bq011
db                                                                  ga4
question              How many pseudo users were active in the last ...
external_knowledge            ga4_obfuscated_sample_ecommerce.events.md
temporal                                                           None
Name: 0, dtype: object
{
  "request_payload": {
    "user_id": "user_123",
    "model": "google/gemma-4-E4B-it",
    "first_user_message": "I want to ask a question about my database schema.",
    "question": "How many pseudo users were active in the last 7 days but inactive in the last 2 days as of January 7, 2021?",
    "is_thinking": true,
    "skip_user_interaction": true,
    "schema_context_base": "CREATE TABLE `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_20210109`\n(\n  event_date STRING,\n  event_timestamp INT64,\n  event_name STRING,\n  event_params ARRAY<STRUCT<key STRING, value STRUCT<string_value STRING, 

In [131]:
sample_result["ask_answer"]

"SELECT\n  COUNT(DISTINCT t1.user_pseudo_id)\nFROM bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_20210109 AS t1\nWHERE\n  t1.event_date BETWEEN '2020-12-31' AND '2021-01-07' AND t1.user_pseudo_id IN (\n    SELECT\n      user_pseudo_id\n    FROM bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_20210109\n    WHERE\n      event_date BETWEEN '2020-12-31' AND '2021-01-07'\n    GROUP BY\n      user_pseudo_id\n    HAVING\n      COUNT(DISTINCT CASE WHEN event_date IN ('2021-01-06', '2021-01-07') THEN 1 END) = 0 AND COUNT(DISTINCT CASE WHEN event_date BETWEEN '2020-12-31' AND '2021-01-05' THEN 1 END) > 0\n  )"

In [135]:
query_job = client.query(sample_result["ask_answer"], location="US")
rows = query_job.result()

In [136]:
for row in rows:
    print(row)

Row((0,), {'f0_': 0})


In [129]:
#ask_llm()